In [ ]:
# ====================================
# Import
# ====================================
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

import joblib

def evaluate_pr(model, X_test, y_test):

    prob = model.predict_proba(X_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_test,
        prob
    )

    ap = average_precision_score(
        y_test,
        prob
    )

    print(ap)

    return precision, recall, ap

def plot_pr(baseline, precision, recall, ap):

    plt.plot(
        recall,
        precision,
        label=f"AP = {ap:.2f}"
    )

    plt.axhline(
        y = baseline,
        linestyle="--",
        label="Random baseline"
    )

    plt.xlabel(
        "Recall"
    )

    plt.ylabel(
        "Precision"
    )

    plt.title(
        "Precision-Recall Curve"
    )

    plt.legend()
    plt.show()


# ====================================
# Load
# ====================================

BASE_DIR = Path.cwd()

ini_file = BASE_DIR / "data" / "employees.csv"

df = pd.read_csv(ini_file)

# ====================================
# Data Cleaning
# ====================================

df["PerformanceScore"] = df["PerformanceScore"].fillna(
    df["PerformanceScore"].mode()[0]
)
df["Education"] = df["Education"].fillna(
    df["Education"].mode()[0]
)

# ====================================
# Creating unbalanced data
# ====================================

df["HighSalary"] = (
    df["Salary"] >= df["Salary"].median()
)

df_false = df[
    df["HighSalary"] == False
]

df_true = df[
    df["HighSalary"] == True
]

df_true = df_true.sample(
    n=10,
    random_state=42
)

df = pd.concat(
    [
        df_false,
        df_true
    ]
)

# ====================================
# Feature Selection
# ====================================

X = df[[
    "PerformanceScore",
    "Education",
    "Department",
    "Experience"
]].copy()


# ====================================
# Feature Engineering
# ====================================

X["ExperienceScore"] = (
    X["Experience"] * X["PerformanceScore"]
)

# ====================================
# Target
# ====================================

y = df["HighSalary"]

# ====================================
# Train/Test Split
# ====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ====================================
# Feature split
# ====================================

numeric_features = [
    "PerformanceScore",
    "Experience",
    "ExperienceScore"
]

categorical_features = [
    "Education",
    "Department"
]

# ====================================
# ColumnTransformer
# ====================================

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(drop="first"), 
        categorical_features
    )
])

# ====================================
# Pipeline and pipeline usage
# ====================================

pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced"
        )
    )
])

pipeline.fit(X_train, y_train)

model = pipeline.named_steps["model"]

model_path = BASE_DIR / "models" / "logistic_regression.pkl"

joblib.dump(
    pipeline,
    model_path
)

# ====================================
# Print model coeficients
# ====================================

feature_names = (
    pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    pipeline
    .named_steps["model"]
    .coef_[0]
)

# ====================================
# Odds ratio
# ====================================

odds_ratio = np.exp(
    coefficients
)

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Odds ratio": odds_ratio
})

# ====================================
# AbsCoefficient
# ====================================

coef_df["AbsCoefficient"] = (
    coef_df["Coefficient"].abs()
)

coef_df = coef_df.sort_values(
    "AbsCoefficient",
    ascending=False
)

print(coef_df)



                               Feature  Coefficient  Odds ratio  \
7           categorical__Department_IT     0.741895    2.099910   
3   categorical__Education_High School    -0.639844    0.527375   
9   categorical__Department_Operations    -0.633168    0.530907   
10       categorical__Department_Sales     0.631791    1.880977   
5           categorical__Education_PhD    -0.429034    0.651138   
0            numeric__PerformanceScore     0.423698    1.527599   
4        categorical__Education_Master     0.304870    1.356448   
1                  numeric__Experience     0.293527    1.341149   
8    categorical__Department_Marketing    -0.279462    0.756191   
2             numeric__ExperienceScore     0.170065    1.185382   
6           categorical__Department_HR    -0.076322    0.926518   

    AbsCoefficient  
7         0.741895  
3         0.639844  
9         0.633168  
10        0.631791  
5         0.429034  
0         0.423698  
4         0.304870  
1         0.293527  
8     